In [1]:
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')   # Required by WordNet for newer NLTK versions
nltk.download('punkt')     # If you're tokenizing text

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vineet\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vineet\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\vineet\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vineet\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [4]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
26,"The script for ""Scary Movie 2"" just wasn't rea...",negative
911,"Seeing this movie, as I just did for the first...",negative
852,When one considers that Carson McCullers is on...,negative
717,Having majored in Western History when I was a...,positive
996,I first saw this movie as a younger child. My ...,positive


In [5]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [6]:
df = normalize_text(df)
df.head()

,review,sentiment
26,script scary movie ready go problem film blata...,negative
911,seeing movie first time turner classic which l...,negative
852,one considers carson mccullers one foremost li...,negative
717,majored western history student college seeing...,positive
996,first saw movie younger child sister told thou...,positive


In [7]:
df['sentiment'].value_counts()

sentiment
negative    261
positive    239
Name: count, dtype: int64

In [8]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [9]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
26,script scary movie ready go problem film blata...,0
911,seeing movie first time turner classic which l...,0
852,one considers carson mccullers one foremost li...,0
717,majored western history student college seeing...,1
996,first saw movie younger child sister told thou...,1


In [10]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [19]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/vineetc2/Capstone-project.mlflow')
dagshub.init(repo_owner='vineetc2', repo_name='Capstone-project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2026-07-05 12:56:25,559 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/vineetc2/Capstone-project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "vineetc2/Capstone-project"

2026-07-05 12:56:25,577 - INFO - Initialized MLflow to track repo "vineetc2/Capstone-project"


Repository vineetc2/Capstone-project initialized!

2026-07-05 12:56:25,583 - INFO - Repository vineetc2/Capstone-project initialized!


<Experiment: artifact_location='mlflow-artifacts:/85fc7146883b436ebe5be81094d89d2e', creation_time=1783235637301, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1783235637301, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [23]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.2)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-07-05 13:06:06,077 - INFO - Starting MLflow run...
2026-07-05 13:06:07,201 - INFO - Logging preprocessing parameters...
2026-07-05 13:06:08,155 - INFO - Initializing Logistic Regression model...
2026-07-05 13:06:08,170 - INFO - Fitting the model...
2026-07-05 13:06:08,226 - INFO - Model training complete.
2026-07-05 13:06:08,227 - INFO - Logging model parameters...
2026-07-05 13:06:08,540 - INFO - Making predictions...
2026-07-05 13:06:08,541 - INFO - Calculating evaluation metrics...
2026-07-05 13:06:08,556 - INFO - Logging evaluation metrics...
2026-07-05 13:06:09,752 - INFO - Saving and logging the model...
2026/07/05 13:06:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-07-05 13:06:29,419 - INFO - Model training and logging completed in 22.22 seconds.
2026-07-05 13:06:29,420 - INFO - Accuracy: 0.64
2026-07-05 13:06:29,421 - INFO - Precision: 0.5625
2026-07-05 13:06:29,423 - INFO - Recall: 0.6428571428571429
2026-07-05 13:06:29,423

🏃 View run rare-wren-117 at: https://dagshub.com/vineetc2/Capstone-project.mlflow/#/experiments/0/runs/bd634faba6c6467487415a6acbcb97a5
🧪 View experiment at: https://dagshub.com/vineetc2/Capstone-project.mlflow/#/experiments/0
